# 03_xgboost_cv_optuna.ipynb

Optimización de hiperparámetros de XGBoost con Optuna usando embeddings.
- Entrada: `../backend/models/embeddings_arrays.npz` (metadatos en `embeddings.pkl`)
- Convertimos a HDF5 (una sola vez) -> `../backend/models/embeddings.h5`
- Lectura por slices desde HDF5 para evitar OOM
- Salida: `../backend/models/optuna_study.pkl`, `optuna_best_params.pkl`, `optuna_best_params.json`, `optuna_cv_scores.pkl`


# Celda 1 — Imports y rutas 

In [1]:
# Celda 1 - imports y paths
import os, time, json, pickle
from pathlib import Path

import numpy as np
import optuna
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# Rutas
NOTEBOOKS_DIR = Path.cwd()
REPO_ROOT = NOTEBOOKS_DIR.parent.resolve()
MODELS_DIR = REPO_ROOT / "backend" / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)
print("REPO_ROOT:", REPO_ROOT)
print("MODELS_DIR:", MODELS_DIR)


c:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NOTEBOOKS_DIR: c:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\notebooks
REPO_ROOT: C:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2
MODELS_DIR: C:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\backend\models


# Celda 2 — Comprobar fichero .npz y embeddings.pkl 

In [2]:
# Celda 2 - comprobar que existen los artefactos previos
npz_path = MODELS_DIR / "embeddings_arrays.npz"
meta_pkl = MODELS_DIR / "embeddings.pkl"

if not meta_pkl.exists():
    raise FileNotFoundError(f"No se encontró metadata embeddings: {meta_pkl}")
if not npz_path.exists():
    raise FileNotFoundError(f"No se encontró arrays npz: {npz_path}")

print("Found:", npz_path)
print("Metadata:", meta_pkl)


Found: C:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\backend\models\embeddings_arrays.npz
Metadata: C:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\backend\models\embeddings.pkl


In [4]:
# Celda debug: inspeccionar contenido del npz
import numpy as np
from pathlib import Path

npz_path = Path("../backend/models/embeddings_arrays.npz")
with np.load(npz_path, mmap_mode="r") as arrays:
    print("Keys:", list(arrays.keys()))
    X = arrays["embeddings"]
    y = arrays["y"]
    print("Embeddings dtype:", X.dtype)
    print("Embeddings ndim:", X.ndim)
    print("Embeddings shape:", X.shape)
    print("y shape:", y.shape)


Keys: ['embeddings', 'y']
Embeddings dtype: float16
Embeddings ndim: 4
Embeddings shape: (20760, 224, 224, 3)
y shape: (20760,)


# 🟩 Celda 3 — Convertir .npz -> embeddings.h5 (ejecutar solo 1 vez) 

In [ ]:
# Celda 3 - convertir a HDF5 (rápido, robusto y permite lectura por slices)
import h5py

arrays_npz = npz_path
h5_path = MODELS_DIR / "embeddings.h5"

# Ajusta CHUNK_SIZE si tu RAM es limitada (ej. 500-2000)    
CHUNK_SIZE = 2000

if not h5_path.exists():
    print("Convirtiendo", arrays_npz, "→", h5_path, " con chunk_size=", CHUNK_SIZE)
    with np.load(arrays_npz, mmap_mode='r') as arrays:
        X_src = arrays['embeddings']
        y_src = arrays['y']
        n, d = X_src.shape
        print(f"N={n}, D={d}")
        with h5py.File(h5_path, "w") as hf:
            X_ds = hf.create_dataset("embeddings", shape=(n, d), dtype="float32",
                                     chunks=(min(CHUNK_SIZE, n), d), compression="gzip")
            y_ds = hf.create_dataset("y", shape=(n,), dtype="int32",
                                     chunks=(min(CHUNK_SIZE, n)), compression="gzip")
            i = 0
            while i < n:
                j = min(i + CHUNK_SIZE, n)
                print(f"  copying rows {i}..{j}")
                X_ds[i:j] = X_src[i:j].astype("float32")
                y_ds[i:j] = y_src[i:j].astype("int32")
                i = j
    print("✅ Conversión HDF5 completada:", h5_path)
else:
    print("HDF5 ya existe:", h5_path)


Convirtiendo C:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\backend\models\embeddings_arrays.npz → C:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\backend\models\embeddings.h5  con chunk_size= 2000


ValueError: too many values to unpack (expected 2)

# 🟩Celda 4 — Abrir HDF5 (mantener abierto) y chequear device GPU (Código)

In [ ]:
# Celda 3 - device check and basic options
import torch
has_cuda = torch.cuda.is_available()
print("torch.cuda.is_available():", has_cuda)
if has_cuda:
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected - running on CPU")

# We will set GPU params conditionally in the objective.


# 🟩 Celda 4 — Objetivo Optuna 

In [ ]:
# Celda 4 - abrir HDF5 (no carga en memoria) y comprobar GPU
import h5py
import torch

h5_path = MODELS_DIR / "embeddings.h5"
assert h5_path.exists(), "Ejecuta la celda de conversión (.npz -> .h5) primero."

hf = h5py.File(h5_path, "r")
X_ds = hf["embeddings"]
y_ds = hf["y"]

print("HDF5 abierto:")
print("  embeddings:", X_ds.shape, X_ds.dtype)
print("  labels:", y_ds.shape, y_ds.dtype)

has_cuda = torch.cuda.is_available()
print("GPU disponible:", has_cuda)
if has_cuda:
    print("GPU name:", torch.cuda.get_device_name(0))


# 🟩 Celda 5 — Definir objective (usa slices desde HDF5) 

In [ ]:
# Celda 5 - definición del objective para Optuna (usa slices por fold)
RANDOM_SEED = 42

def make_xgb_params(trial, use_cuda: bool):
    params = {
        "verbosity": 0,
        "objective": "multi:softprob",
        "num_class": int(len(np.unique(y_ds[:]))),  # lectura mínima para saber num clases
        "eval_metric": "mlogloss",
        "eta": float(trial.suggest_float("learning_rate", 1e-3, 0.3, log=True)),
        "max_depth": int(trial.suggest_int("max_depth", 3, 10)),
        "min_child_weight": float(trial.suggest_float("min_child_weight", 1, 10)),
        "subsample": float(trial.suggest_float("subsample", 0.5, 1.0)),
        "colsample_bytree": float(trial.suggest_float("colsample_bytree", 0.5, 1.0)),
        "gamma": float(trial.suggest_float("gamma", 0.0, 5.0)),
        "lambda": float(trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True)),
        "alpha": float(trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True)),
        "tree_method": "hist"   # compatible en CPU y GPU
    }
    if use_cuda:
        params["device"] = "cuda"
    return params

def objective(trial):
    use_cuda = has_cuda
    params = make_xgb_params(trial, use_cuda=use_cuda)
    num_boost_round = int(trial.suggest_int("n_estimators", 100, 800))
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    scores = []
    
    # NOTA: y_ds[:] carga todo, pero solo para np.unique; para folds usamos slices
    y_all = y_ds[:]  # etiquetas en memoria (int32) — suele ser pequeño comparado con X
    for train_idx, val_idx in skf.split(np.zeros(len(y_all)), y_all):
        # Leer solo las filas necesarias del HDF5
        X_train = X_ds[train_idx, :]   # lee porciones desde disco
        y_train = y_all[train_idx]
        X_val = X_ds[val_idx, :]
        y_val = y_all[val_idx]
        
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval = xgb.DMatrix(X_val, label=y_val)
        
        bst = xgb.train(
            params,
            dtrain,
            num_boost_round=num_boost_round,
            evals=[(dtrain, "train"), (dval, "valid")],
            early_stopping_rounds=30,
            verbose_eval=False
        )
        preds = np.argmax(bst.predict(dval), axis=1)
        acc = accuracy_score(y_val, preds)
        scores.append(acc)
    return float(np.mean(scores))


# 🟩 Celda 6 — Ejecutar Optuna (Código)

In [ ]:
# Celda 6 - ejecutar Optuna
N_TRIALS = 30   # prueba con 30; súbelo a 50+ si tu máquina lo aguanta
print("Running Optuna with N_TRIALS =", N_TRIALS, " — this may take time")

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))

t0 = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
t1 = time.time()

print("Optuna done — time (s):", round(t1-t0,1))
print("Best CV accuracy:", study.best_value)
print("Best params:", study.best_params)


# 

# 🟩 Celda 7 — Guardar estudio y summary (pkl + json) (Código)

In [ ]:
# Celda 7 - guardar artefactos de Optuna
study_path = MODELS_DIR / "optuna_study.pkl"
best_path = MODELS_DIR / "optuna_best_params.pkl"
best_json = MODELS_DIR / "optuna_best_params.json"

with open(study_path, "wb") as f:
    pickle.dump(study, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(best_path, "wb") as f:
    pickle.dump({"best_value": float(study.best_value), "best_params": study.best_params}, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(best_json, "w", encoding="utf-8") as f:
    json.dump({"best_value": float(study.best_value), "best_params": study.best_params}, f, indent=2, ensure_ascii=False)

print("Saved study:", study_path)
print("Saved best params (pkl):", best_path)
print("Saved best params (json):", best_json)


# 🟩 Celda 8 — Evaluación rápida con los mejores params (guardado scores) (Código)

In [ ]:
# Celda 8 - evaluar el mejor conjunto (3-fold) y guardar scores
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np

best = {"best_value": float(study.best_value), "best_params": study.best_params}

# extraemos parámetros (y forzamos tree_method/device coherentes)
best_params = best["best_params"].copy()
best_params["tree_method"] = "hist"
if has_cuda:
    best_params["device"] = "cuda"

num_boost_round = int(best_params.pop("n_estimators", 300)) if "n_estimators" in best_params else 300

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
y_all = y_ds[:]
cv_scores = []
for train_idx, val_idx in skf.split(np.zeros(len(y_all)), y_all):
    X_train = X_ds[train_idx, :]
    y_train = y_all[train_idx]
    X_val = X_ds[val_idx, :]
    y_val = y_all[val_idx]
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    bst = xgb.train(best_params, dtrain, num_boost_round=num_boost_round, evals=[(dtrain,"train")], verbose_eval=False)
    preds = np.argmax(bst.predict(dval), axis=1)
    acc = accuracy_score(y_val, preds)
    cv_scores.append(acc)

print("CV scores with best params:", cv_scores)
print("Mean CV:", float(np.mean(cv_scores)))

scores_path = MODELS_DIR / "optuna_cv_scores.pkl"
with open(scores_path, "wb") as f:
    pickle.dump({"cv_scores": cv_scores, "mean": float(np.mean(cv_scores))}, f, protocol=pickle.HIGHEST_PROTOCOL)
print("Saved CV scores to:", scores_path)


# Celda 9 — (Opcional) Entrenar modelo final sobre TODO y guardar (Código)

In [ ]:
# Celda 9 - Entrenar modelo final en todo el dataset y guardar booster (opcional)
train_final = False  # cambia a True si quieres entrenar y guardar el booster final

if train_final:
    print("Training final model on full dataset...")
    final_params = best_params.copy()  # ya sin n_estimators
    num_boost_round = num_boost_round if 'num_boost_round' in locals() else 300

    dtrain_all = xgb.DMatrix(X_ds[:], label=y_ds[:])
    final_bst = xgb.train(final_params, dtrain_all, num_boost_round=num_boost_round, evals=[(dtrain_all,"train")], verbose_eval=False)

    # Guardar JSON nativo del booster + wrapper pkl pequeño
    final_json = MODELS_DIR / "xgb_final_model.json"
    final_pkl = MODELS_DIR / "xgb_final_model_from_optuna.pkl"
    final_bst.save_model(str(final_json))
    with open(final_pkl, "wb") as f:
        pickle.dump({"model_json": str(final_json)}, f, protocol=pickle.HIGHEST_PROTOCOL)
    print("Saved final model JSON:", final_json)
    print("Saved wrapper pkl:", final_pkl)


# Celda 10 — Cerrar HDF5 y limpieza (Código)

In [ ]:
# Celda 10 - cerrar HDF5
try:
    hf.close()
    print("HDF5 cerrado.")
except Exception as e:
    print("Warning closing HDF5:", e)


# Celda 11 — Notas y recomendaciones (Markdown)

### Notas y recomendaciones

- Optuna puede tardar mucho según `N_TRIALS` y `n_estimators`. Empieza con 12-30 pruebas y sube si el tiempo lo permite.
- Si usas GPU, asegúrate de que XGBoost está compilado con soporte CUDA. En el caso en que `device='cuda'` falle, cambia por CPU o recompila XGBoost con GPU.
- Usa `train_final=True` solo cuando estés seguro del mejor conjunto (y tengas tiempo). Guardará `xgb_final_model.json` en `../backend/models`.
- Si necesitas ahorrar aún más RAM durante entrenamiento CV, reduce `n_splits` a 3 (ya está así) o reduce `num_boost_round` en la búsqueda.
